<a href="https://colab.research.google.com/github/MNIKIEMA/differentiable-wonderland-lab-sessions/blob/main/Lab_JAX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Guided lab session

### Topic: Logistic regression in JAX

The objective of this lab is to introduce the main concepts of the [JAX](https://jax.readthedocs.io/) library via a simple implementation of a logistic regression on a toy dataset. We discuss most of the *functional transformations* provided by the library (`jit`, `grad`, `vmap`), as well as some core implementation details (jaxprs, pytrees, ...).

> 🟡 Before starting, I strongly suggest looking at the [quickstart](https://jax.readthedocs.io/en/latest/quickstart.html) in the library's documentation. I assume you are familiar with working with $n$-dimensional arrays in NumPy.

In [1]:
# We use jaxtyping to annotate shapes
# https://docs.kidger.site/jaxtyping/api/array/
%pip install jaxtyping --quiet
from jaxtyping import Array, Int, Float
from typing import Tuple

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.4 MB/s eta 0:00:00


### Section 1: Linear algebra in JAX

In [2]:
import jax
import jax.numpy as jnp
import numpy as np

In [3]:
# The jax.numpy module is a replica of the NumPy API, with a few
# notable exceptions: https://jax.readthedocs.io/en/latest/jax.numpy.html
A = jnp.ones((3, 4))
B = A.T @ A

In [4]:
# Arrays in JAX / NumPy are characterized by their dtype and their shape.
print(A.shape)
print(A.dtype)

(3, 4)
float32


In [5]:
# Everything in JAX is (supposed to be) a pure function, meaning a function
# with no side-effects (https://en.wikipedia.org/wiki/Pure_function).
# As an example, the PNRG module in NumPy does not satisfy this property, as the
# internal state changes everytime we request new values.
np.random.seed(0)
print(np.random.get_state()[2])
np.random.normal(size=(3, 2))
print(np.random.get_state()[2])

624
12


In [6]:
# In order to "purify" the PRNG module, we explicitly instantiate and propagate
# the internal state, represented as a "key" (note: I will not go in detail on
# what the key represents, lots of info on the library's documentation if you are
# interested in reading more).
key = jax.random.PRNGKey(0)

In [7]:
# Whenever we need to sample some data, we "split" the key to obtain another key
# to be used only once. Note how side effects have disappeared since everything
# related to the state is handled explicitly with the keys.
key, subkey = jax.random.split(key)
print(jax.random.normal(subkey, shape=(3, 2)))
# print(jax.random.normal(subkey, shape=(3, 2)))

[[-2.4424558  -2.0356805 ]
 [ 0.20554423 -0.3535502 ]
 [-0.76197404 -1.1785518 ]]


In [8]:
# Note the process: key --> key ------> key
#                        |-> subkey |
#                                   |-> subkey
key, subkey = jax.random.split(key)
print(jax.random.normal(subkey, shape=(3, 2)))

[[-1.2574776  -0.4016044 ]
 [-1.1213601   0.87837774]
 [-0.86175495  0.34651348]]


> **TODO**: consider a generic logistic regression model $\mathbf{y} = \text{softmax}(\mathbf{W}\mathbf{x} + \mathbf{b})$. Complete the following function to initialize the parameters.

In [9]:
def init_params(key: jax.random.PRNGKey, num_features: int, num_classes: int) -> Tuple[Float[Array, "num_features num_classes"], Float[Array, "num_classes"]]:
  # TODO: complete the function.
  key, subkey = jax.random.split(key)
  W = jax.random.normal(subkey, shape=(num_features, num_classes))
  b = jax.random.normal(subkey, shape=(num_features))
  return W, b

In [10]:
key, subkey = jax.random.split(key)
W, b = init_params(subkey, 5, 3)
assert(W.shape == (5, 3))

### Interlude: Loading a dataset

Differently from TensorFlow, PyTorch, etc., JAX does not have many high-level utilities related to deep learning (data loading, model's definition, etc.). This is actually a flexibility, since we can use any known data loader in our programs. For this lab, we consider a simple drop-in replacement of the Iris dataset (**penguins**) found on [TensorFlow Hub](https://www.tensorflow.org/hub). There are no exercises in this section, see the labs in TensorFlow for an introduction.

In [11]:
import tensorflow_datasets as tfds
from sklearn.model_selection import train_test_split

In [12]:
data = tfds.load('penguins', as_supervised=True)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/penguins/processed/incomplete.7W8U8U_1.0.0/penguins-train.tfrecord*...:   …

Dataset penguins downloaded and prepared to /root/tensorflow_datasets/penguins/processed/1.0.0. Subsequent calls will reuse this data.


In [13]:
data

{'train': <_PrefetchDataset element_spec=(TensorSpec(shape=(4,), dtype=tf.float32, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>}

In [14]:
# Like PyTorch's data loaders, TensorFlow datasets are built to allow iterations
# over very large datasets. We ignore this aspect here and we simply extract the
# input and output matrices.
X, y = data['train'].batch(10000).get_single_element()

In [15]:
print(X.shape)
print(y.shape)

(334, 4)
(334,)


In [16]:
# No more TensorFlow from here onwards. :-)
X, y = X.numpy(), y.numpy()

In [17]:
# Split into training and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

In [18]:
NUM_FEATURES = X_train.shape[1]
NUM_CLASSES = np.unique(y_train).shape[0]

### Section 2: Efficient compilation with `jax.jit`

The core of JAX is composed of *functional transformations*, which operate on pure JAX function and return other JAX functions (thus allowing for composition). The first we introduce is `jax.jit` for compiling the code to an efficient version.

> **TODO**: Implement a function to perform one-hot encoding of the target values. We will use the function to benchmark the compilation. Remember that, given an index $i$ and a number of classes $c$, a one-hot encoding is a binary vector of size $c$ such that all values are $0$ except the $i$-th one, which is $1$.

In [19]:
def to_onehot(y: Int[Array, "n"], num_classes: int) -> Int[Array, "n num_classes"]:
  # TODO: Complete the function.
  return jnp.identity(num_classes)

In [20]:
to_onehot(y[0:10], 3)

Array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]], dtype=float32)

JIT (just-in-time) compilation is a procedure to trace the execution of a function and optimize its low-level implementation. This is obtained by defining "abstract" values for the parameters that are used to trace out the behaviour. Read more here: https://jax.readthedocs.io/en/latest/jit-compilation.html

In [21]:
# Execution in JAX is asynchronous, which is why we need the blocking operation
%timeit to_onehot(y, 3).block_until_ready()

231 µs ± 8.11 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [22]:
# JIT does not work well with constants, so we fix one of the parameters with functools.partial
from functools import partial
to_onehot_3 = partial(to_onehot, num_classes=NUM_CLASSES)

In [23]:
# A jaxpr is a sequence of low-level operations implemented in a language
# which is intermediate between the JAX API and the actual instructions executed on the
# hardware. This simplifies compilation and also dispatching to multiple types of hardware.
# make_jaxpr is also a functional transformation, return a function to compute the jaxpr
# of a given function.
jax.make_jaxpr(to_onehot_3)(y)

{ lambda ; a:i32[334]. let
    b:i32[3,3] = iota[dimension=0 dtype=int32 shape=(3, 3) sharding=None] 
    c:i32[3,3] = iota[dimension=1 dtype=int32 shape=(3, 3) sharding=None] 
    d:i32[3,3] = add b 0:i32[]
    e:bool[3,3] = eq d c
    f:f32[3,3] = convert_element_type[new_dtype=float32 weak_type=False] e
  in (f,) }

In [24]:
to_onehot_3_jitted = jax.jit(to_onehot_3)

In [25]:
%timeit to_onehot_3_jitted(y).block_until_ready()

8.38 µs ± 1.69 µs per loop (mean ± std. dev. of 7 runs, 100000 loops each)


### Section 3: pytrees

Any combination of Python containers (dict, list, tuples, ...) in JAX corresponds to a PyTree object (https://jax.readthedocs.io/en/latest/pytrees.html). This is useful because all functional transformations (e.g., JIT) work natively on all pytrees.

In [26]:
# Let us reinitialize the parameters now that we have a dataset
params = init_params(key, NUM_FEATURES, NUM_CLASSES)

In [27]:
# params is a valid PyTree
jax.tree_util.tree_structure(params)

PyTreeDef((*, *))

Moving parameters around in tuples or dictionary can become cumbersome. Python has a [dataclass](https://docs.python.org/3/library/dataclasses.html) object to act as a structured container for information, and JAX allows you to [register new objects](https://jax.readthedocs.io/en/latest/pytrees.html#extending-pytrees) as pytrees, including [dataclasses](https://jax.readthedocs.io/en/latest/_autosummary/jax.tree_util.register_pytree_node_class.html).

> **TODO**: Implement a `dataclass` to keep track of the parameters of the logistic regression module and register it as a valid pytree. This is a good exercise for when you move on to high-level libraries (e.g., Equinox).

In [ ]:
from dataclasses import dataclass

@dataclass
class LogisticRegressionParams:
    # TODO: Complete the data class.

In [ ]:
# Check this is a valid PyTree now
jax.tree_util.tree_structure(LogisticRegressionParams(params[0], params[1]))

> **TODO**: Very quickly, modify the initialization from before to return our new custom node.

In [ ]:
def init_params_2(key: jax.random.PRNGKey, num_features: int, num_classes: int) -> LogisticRegressionParams:
  # TODO: complete the function.
  return ...

### Section 4: vmap (automatic vectorization)

The second function we introduce is [jax.vmap](https://jax.readthedocs.io/en/latest/_autosummary/jax.vmap.html), which is used to automatically vectorize the code over new axes. Consider the following implementation of logistic regression:

In [ ]:
def call_fcn(params: LogisticRegressionParams, X: Float[Array, "n num_features"]) -> Float[Array, "n num_classes"]:
  return jax.nn.softmax(jnp.dot(X, params[0]) + params[1], axis=1)

A better option in JAX is to define the function for a single sample (a single row of $\mathbf{X}$) and use `jax.vmap` to add the new axis.

> **TODO**: Implement the vmapped variant of `call_fcn`.

In [ ]:
def call_fcn(params: LogisticRegressionParams, x: Float[Array, "num_features"]) -> Float[Array, "num_classes"]:
  # TODO: Complete the function.
  return ...

In [ ]:
# Equivalent calls to understand decorators and partial.
# call_fcn = jax.vmap(call_fcn, in_axes=(None, 0), out_axes=0)
# call_fcn = partial(jax.vmap, in_axes=(None, 0))(call_fcn)

In [ ]:
# jax.vmap with a single input:
# f(x)   vmap(f) := f([x1, x2, ..., xn]) -> [f(x1), f(x2), ..., f(xn)]

# jax.vmap with two inputs:
# f(x, y) vmap(f) := f([x1, ..., xn], [y1, ..., yn]) =[f(x1, y1), ..., f(xn, yn)]

# jax.vmap with two inputs, only one of which is vectorized:
# f(x, y) vmap(f, in_axes=(None, 0)) := f(x, [y1, ..., yn]) = [f(x, y1), ..., f(x, yn)]

In [ ]:
assert(call_fcn(params, X).shape == (X.shape[0], NUM_CLASSES))

### 5. Training the model

> **TODO**: Write a function to compute the accuracy of a given set of predictions.

In [ ]:
def accuracy(y: Float[Array, "n"], y_pred: Float[Array, "n 3"]) -> Float[Array, ""]:
  # TODO: Complete the function.
  return ...

Recall that the cross-entropy loss is given by $l(y, \hat{y}) = - \sum_c y_c \log(\hat{y}_c)$, where the sum is over the classes and $y$ is assumed as a one-hot vector. Alternatively, $l(y, \hat{y}) = - \log(\hat{y}_y)$, where $y$ is the index of the correct class.

> **TODO**: Implement the cross-entropy loss, with a `jax.vmap` step for batching.

In [ ]:
def cross_entropy(y: Float[Array, "num_classes"], y_pred: Float[Array, "num_classes"]) -> Float[Array, ""]:
  # TODO: Complete the function.
  return ...

In [ ]:
# value_and_grad is a functional transformation that converts loss into a function
# returning two values: the original output and the gradients of the output with
# respect to the first input parameter.
@jax.jit
@jax.value_and_grad
def loss(params):
  return cross_entropy(to_onehot_3(y), call_fcn(params, X)).mean()

To conclude, we implement a simple gradient descent procedure (in practice, it would be better to `jax.jit` the entire training step, which we ignore here for simplicity):

$$ w_t = w_{t-1} - \eta \nabla L(w_{t-1}) $$

In [ ]:
params = init_params(key, NUM_FEATURES, NUM_CLASSES)
loss_history = []

for i in range(1000):
   l, g = loss(params)
   loss_history.append(l)
   # Because everything is a PyTree, we can map over them to update all
   # values with the corresponding gradients.
   params = jax.tree.map(lambda p, grad_p: p - 0.01*grad_p, params, g)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(loss_history)

In [ ]:
accuracy(y_test, call_fcn(params, X_test))

As a final exercise, consider a modified variant of gradient descent where we insert a momentum term:

\begin{align*}
g_t & =- \eta \nabla L(w_{t-1}) + \lambda g_{t-1} \\
w_{t}& = w_{t-1}+ g_t
\end{align*}

> **TODO**: Implement the gradient descent with momentum procedure and compare the results with the previous run.

In [ ]:
# TODO: Write the code.
loss_history_momentum = ...

In [ ]:
import matplotlib.pyplot as plt
plt.plot(loss_history, label='Standard')
plt.plot(loss_history_momentum, label='Momentum')
plt.legend()